In [1]:
import pandas as pd

events = pd.read_parquet("../data/raw/events.parquet")
lineups = pd.read_parquet("../data/raw/lineups.parquet")

def time_to_seconds(time_str):
    """
    Converts StatsBomb time strings like '83:10'
    into total seconds.
    """
    minutes, seconds = map(int, time_str.split(":"))
    return minutes * 60 + seconds

In [2]:
profiles = pd.read_parquet("../data/processed/player_profiles.parquet")

profiles.info()
profiles.head(5)

<class 'pandas.DataFrame'>
RangeIndex: 255 entries, 0 to 254
Data columns (total 34 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   player_id                         255 non-null    int64  
 1   player_name                       255 non-null    str    
 2   competition_name                  255 non-null    str    
 3   season                            255 non-null    str    
 4   role                              255 non-null    str    
 5   lineup_share                      255 non-null    float64
 6   event_share                       255 non-null    float64
 7   deep_progressions                 255 non-null    int64  
 8   open_play_xg_assisted             255 non-null    float64
 9   turnovers                         255 non-null    float64
 10  aerial_win_pct                    245 non-null    float64
 11  padj_tackles_interceptions        255 non-null    float64
 12  padj_pressures     

,player_id,player_name,competition_name,season,role,lineup_share,event_share,deep_progressions,open_play_xg_assisted,turnovers,...,open_play_xg_assisted_per90,turnovers_per90,padj_tackles_interceptions_per90,padj_pressures_per90,fouls_per90,touches_in_box_per90,shots_per90,npxg_per90,fouls_won_per90,profile_id
0,27148,Abraham González Casanova,La Liga,2015/2016,Creative Midfielder,0.428571,0.595404,422,2.334519,17.0,...,0.182701,1.330435,3.005932,1.139707,1.095652,2.895652,1.252174,0.221824,0.704348,e185870b-7fad-5b21-a823-eb16db9e6b53
1,6557,Adrián González Morales,La Liga,2015/2016,Creative Midfielder,0.581395,0.653197,780,1.205470,46.0,...,0.047397,1.808650,1.889053,0.873123,1.847969,3.381389,1.847969,0.189731,0.668414,2b8f2a23-f1c1-530b-9484-b04feab44e98
2,6926,Alen Halilović,La Liga,2015/2016,Creative Midfielder,0.563636,0.655665,1057,2.951738,90.0,...,0.108965,3.322395,0.699142,0.351399,0.442986,3.027071,2.178015,0.155142,2.178015,ee9eb907-29c0-552e-ba17-0df7603a5316
3,6737,Alfonso Pedraza Sag,La Liga,2015/2016,Creative Midfielder,0.666667,0.937500,30,0.023744,3.0,...,0.020352,2.571429,0.423154,0.423154,2.571429,2.571429,0.857143,0.047480,0.857143,ba7b5f2e-8d74-534f-8cf9-74c598686644
4,10802,André Filipe Tavares Gomes,La Liga,2015/2016,Creative Midfielder,0.775510,0.857961,1488,3.054531,108.0,...,0.111933,3.957655,2.385297,0.847273,2.345277,3.004886,1.539088,0.088020,2.015472,7965e5c9-4108-5d10-bc85-bc77dba454bd


In [4]:
half_end = (
    events.loc[events["type"] == "Half End"]
    .groupby("match_id")[["minute", "second"]]
    .max()
    .reset_index()
)

half_end["match_end_seconds"] = (
    half_end["minute"] * 60
    + half_end["second"]
)

match_end_lookup = dict(
    zip(
        half_end["match_id"],
        half_end["match_end_seconds"]
    )
)

In [47]:
def calculate_player_match_minutes(positions, match_end_seconds):
    """
    Calculates total minutes played in a match
    using StatsBomb position stints.
    """

    if len(positions) == 0:
        return 0.0

    total_seconds = 0

    for stint in positions:

        start_seconds = time_to_seconds(
            stint["from"]
        )

        if stint["to"] is not None:

            end_seconds = time_to_seconds(
                stint["to"]
            )

        else:

            end_seconds = match_end_seconds

        total_seconds += (
            end_seconds - start_seconds
        )

    return total_seconds / 60

In [48]:
minutes_df = lineups.copy()

minutes_df["match_minutes"] = minutes_df.apply(
    lambda row: calculate_player_match_minutes(
        row["positions"],
        match_end_lookup[row["match_id"]]
    ),
    axis=1
)

season_minutes = (
    minutes_df
    .groupby(
        ["player_id", "player_name"],
        as_index=False
    )["match_minutes"]
    .sum()
    .rename(
        columns={
            "match_minutes": "minutes_played"
        }
    )
)

season_minutes["minutes_played"] = (
    season_minutes["minutes_played"]
    .round()
    .astype(int)
)

In [49]:
def search(player_name):

    result = season_minutes[
        season_minutes["player_name"]
        .str.contains(
            player_name,
            case=False,
            na=False
        )
    ].sort_values(
        "minutes_played",
        ascending=False
    )

    return result[
        ["player_name", "minutes_played"]
    ]

In [60]:
search("James David Rodríguez Rubio")

,player_name,minutes_played
70,James David Rodríguez Rubio,1560


In [51]:
audit_df = lineups.copy()

audit_df["played_match"] = audit_df["positions"].apply(
    lambda x: len(x) > 0
)

player_audit = (
    audit_df
    .groupby(["player_id", "player_name"])
    .agg(
        squad_matches=("match_id", "count"),
        appearances=("played_match", "sum"),
        bench_only=("played_match", lambda x: (~x).sum())
    )
    .reset_index()
)

player_audit["difference"] = (
    player_audit["squad_matches"]
    - player_audit["appearances"]
)

player_audit.sort_values(
    "difference",
    ascending=False
).head(50)

,player_id,player_name,squad_matches,appearances,bench_only,difference
566,31478,Balázs Megyeri,38,0,38,38
286,7069,Miguel Ángel Moyà Rumbo,38,0,38,38
150,6662,Iago Herrerín Buisán,38,2,36,36
320,8803,Daniel Giménez Hernández,37,3,34,34
177,6706,Francisco Casilla Cortés,38,4,34,34
548,27462,Xabier Iruretagoiena Aranzamendi,37,4,33,33
185,6722,David Soria Solís,32,0,32,32
392,20055,Marc-André ter Stegen,38,7,31,31
347,11537,Alberto García Cabrera,37,6,31,31
302,7905,Raúl Lizoáin Cruz,38,8,30,30


In [52]:
validation_players = [
    "Jordi Alba",
    "Toni Kroos",
    "James",
    "Bojan",
    "Borja Ekiza",
    "Juan Francisco García García",
    "Kranevitter",
    "Rachid",
    "Samper",
    "Iñigo Barrenetxea",
    "Mikel Rico",
    "Elustondo",
    "Marcos Llorente",
    "Juan Pablo Añor"
]

player_audit[
    player_audit["player_name"]
    .str.contains(
        "|".join(validation_players),
        case=False,
        na=False
    )
]

,player_id,player_name,squad_matches,appearances,bench_only,difference
41,5211,Jordi Alba Ramos,34,31,3,3
60,5574,Toni Kroos,35,32,3,3
70,5695,James David Rodríguez Rubio,29,26,3,3
108,6563,Juan Pablo Añor Acosta,33,29,4,4
148,6656,Mikel Rico Moreno,30,17,13,13
163,6679,Aritz Elustondo Irribaria,31,31,0,0
247,6840,Marcos Llorente Moreno,5,2,3,3
356,12191,Gorka Elustondo Urkola,22,11,11,11
383,18904,Claudio Matías Kranevitter,12,8,4,4
402,21229,Sergi Samper Montaña,3,1,2,2


In [ ]:
lineups[
    lineups["player_name"]
        .str.contains("Gorka Elustondo Urkola")
][["match_id", "positions"]]

,match_id,positions
429,3825869,"[{'position_id': 6, 'position': 'Left Back', '..."
825,3825842,"[{'position_id': 21, 'position': 'Left Wing', ..."
1076,3825838,[]
2787,3825641,[]
2966,3825797,[]
3129,3825793,"[{'position_id': 9, 'position': 'Right Defensi..."
3524,3825783,[]
3776,3825770,"[{'position_id': 11, 'position': 'Left Defensi..."
5396,3825686,[]
5592,3825675,"[{'position_id': 9, 'position': 'Right Defensi..."


In [54]:

def time_to_seconds(time_str):
    minutes, seconds = map(int, time_str.split(":"))
    return minutes * 60 + seconds


half_end = (
    events.loc[events["type"] == "Half End"]
    .groupby("match_id")[["minute", "second"]]
    .max()
)

match_end_lookup = {
    match_id: row["minute"] * 60 + row["second"]
    for match_id, row in half_end.iterrows()
}


def calculate_match_minutes(positions, match_end_seconds):

    if len(positions) == 0:
        return 0

    total_seconds = 0

    for stint in positions:

        start_seconds = time_to_seconds(stint["from"])

        if stint["to"] is not None:
            end_seconds = time_to_seconds(stint["to"])
        else:
            end_seconds = match_end_seconds

        total_seconds += end_seconds - start_seconds

    return total_seconds / 60


kroos_matches = lineups[
    lineups["player_name"] == "Toni Kroos"
].copy()

kroos_matches["minutes"] = kroos_matches.apply(
    lambda row: calculate_match_minutes(
        row["positions"],
        match_end_lookup[row["match_id"]]
    ),
    axis=1
)

kroos_matches["played"] = kroos_matches["positions"].apply(
    lambda x: len(x) > 0
)

display(
    kroos_matches[
        ["match_id", "played", "minutes", "positions"]
    ]
)

,match_id,played,minutes,positions
9,3825739,True,92.033333,"[{'position_id': 10, 'position': 'Center Defen..."
370,3825846,True,60.216667,"[{'position_id': 15, 'position': 'Left Center ..."
1376,3825833,True,92.116667,"[{'position_id': 10, 'position': 'Center Defen..."
1953,3825820,True,92.083333,"[{'position_id': 13, 'position': 'Right Center..."
2747,3825757,True,69.050000,"[{'position_id': 10, 'position': 'Center Defen..."
2996,3825798,True,94.200000,"[{'position_id': 11, 'position': 'Left Defensi..."
3322,3825787,True,93.400000,"[{'position_id': 10, 'position': 'Center Defen..."
3466,3825785,True,94.083333,"[{'position_id': 10, 'position': 'Center Defen..."
3754,3825770,True,92.150000,"[{'position_id': 10, 'position': 'Center Defen..."
3931,3825678,True,93.083333,"[{'position_id': 11, 'position': 'Left Defensi..."


In [4]:
minutes = pd.read_parquet(
    "../data/processed/player_minutes_summary.parquet"
)

minutes[
    minutes["player_name"]
    .str.contains(
        "Toni Kroos|Jordi Alba|James",
        case=False,
        na=False,
    )
].sort_values(
    "minutes_played",
    ascending=False,
)

,player_id,player_name,season,squad_matches,appearances,bench_only,minutes_played
60,5574,Toni Kroos,2015/2016,35,32,3,2817
41,5211,Jordi Alba Ramos,2015/2016,34,31,3,2665
70,5695,James David Rodríguez Rubio,2015/2016,29,26,3,1560


In [5]:
minutes.info()
minutes.head()

<class 'pandas.DataFrame'>
RangeIndex: 601 entries, 0 to 600
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   player_id       601 non-null    int64
 1   player_name     601 non-null    str  
 2   season          601 non-null    str  
 3   squad_matches   601 non-null    int64
 4   appearances     601 non-null    int64
 5   bench_only      601 non-null    int64
 6   minutes_played  601 non-null    int64
dtypes: int64(5), str(2)
memory usage: 51.6 KB


,player_id,player_name,season,squad_matches,appearances,bench_only,minutes_played
0,3023,Yuri Berchiche Izeta,2015/2016,32,21,11,1906
1,3063,Danilo Luiz da Silva,2015/2016,30,24,6,2107
2,3084,Christian Atsu Twasam,2015/2016,15,12,3,425
3,3130,Gaël Kakuta,2015/2016,3,2,1,47
4,3160,Rene Krhin,2015/2016,33,24,9,1509
